In [ ]:
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

plt.rcParams["figure.dpi"] = 150
plt.rcParams["font.family"] = "serif"

In [ ]:
# Load one event from the test set
shard_path = "/share/rcif2/mmangat/data/colliderml/ttbar/test/ttbar_pu200_tracker_hits/train-00046-of-01000.parquet"
data = ak.from_parquet(shard_path)

EVENT_IDX = 0
ev = data[EVENT_IDX]

x   = ak.to_numpy(ev["x"]).astype(float)
y   = ak.to_numpy(ev["y"]).astype(float)
z   = ak.to_numpy(ev["z"]).astype(float)
r   = np.sqrt(x**2 + y**2)
det = ak.to_numpy(ev["detector"]).astype(int)

det_ids = np.unique(det)
print(f"Event {EVENT_IDX}: {len(x):,} hits, detectors: {det_ids}")

In [ ]:
# Colour palette — one colour per detector
cmap = plt.get_cmap("tab10")
det_colours = {d: cmap(i / max(len(det_ids) - 1, 1)) for i, d in enumerate(det_ids)}

# Subsample to keep the plot fast (plot every Nth hit)
STRIDE = 5
mask_s = np.zeros(len(x), dtype=bool)
mask_s[::STRIDE] = True

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

for d in det_ids:
    m = (det == d) & mask_s
    axes[0].scatter(x[m], y[m], s=0.3, color=det_colours[d], alpha=0.4, label=f"Det {d}", rasterized=True)
    axes[1].scatter(z[m], r[m], s=0.3, color=det_colours[d], alpha=0.4, label=f"Det {d}", rasterized=True)

axes[0].set_xlabel("x [mm]")
axes[0].set_ylabel("y [mm]")
axes[0].set_aspect("equal")
axes[0].set_title("Transverse plane (x–y)")
axes[0].grid(alpha=0.2, ls="--")

axes[1].set_xlabel("z [mm]")
axes[1].set_ylabel("r [mm]")
axes[1].set_title("Longitudinal plane (r–z)")
axes[1].grid(alpha=0.2, ls="--")

axes[1].legend(markerscale=6, loc="upper right", fontsize=8)

fig.suptitle(f"ColliderML PU200 ttbar — event {EVENT_IDX} ({len(x):,} hits)")
plt.savefig("detector_overview.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Zoom in on pixel detectors only (det 0, 1, 2)
pixel_ids = [0, 1, 2]
pixel_mask = np.isin(det, pixel_ids)
print(f"Pixel hits: {pixel_mask.sum():,} / {len(x):,} ({100*pixel_mask.mean():.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

for d in pixel_ids:
    m = det == d
    axes[0].scatter(x[m], y[m], s=0.5, color=det_colours[d], alpha=0.5, label=f"Det {d}", rasterized=True)
    axes[1].scatter(z[m], r[m], s=0.5, color=det_colours[d], alpha=0.5, label=f"Det {d}", rasterized=True)

axes[0].set_xlabel("x [mm]")
axes[0].set_ylabel("y [mm]")
axes[0].set_aspect("equal")
axes[0].set_title("Pixel: transverse (x–y)")
axes[0].grid(alpha=0.2, ls="--")

axes[1].set_xlabel("z [mm]")
axes[1].set_ylabel("r [mm]")
axes[1].set_title("Pixel: longitudinal (r–z)")
axes[1].grid(alpha=0.2, ls="--")

axes[1].legend(markerscale=6, loc="upper right", fontsize=9)

fig.suptitle(f"Pixel subdetectors — event {EVENT_IDX}")
plt.savefig("detector_pixel_zoom.pdf", bbox_inches="tight")
plt.show()